# SFT Playground (Supervised Fine-Tuning)

Обучение на инструкциях/чатах с `SFTDataset`.
- Форматы: **instruct** (instruction/input/output) и **chat** (messages).
- Ручной loop с tqdm, маска labels только по ответам ассистента.

In [ ]:
from pathlib import Path
import torch
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
from transformers import AutoTokenizer

from homellm.models.home_model import HomeConfig, HomeForCausalLM
from homellm.training.sft import SFTDataset

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

In [ ]:
# Путь к SFT данным: JSONL с полями instruction/output или messages
SFT_DATA = Path('/app/datasets/sft_train.jsonl')
if not SFT_DATA.exists():
    # Мини-пример: создаём файл с 2 примерами в формате instruct
    SFT_DATA.parent.mkdir(parents=True, exist_ok=True)
    examples = [
        {"instruction": "Сколько будет 2+2?", "output": "2+2=4."},
        {"instruction": "Переведи на английский: Привет.", "output": "Hello."},
    ]
    import json
    with open(SFT_DATA, 'w', encoding='utf-8') as f:
        for ex in examples:
            f.write(json.dumps(ex, ensure_ascii=False) + '\n')
    print('Created', SFT_DATA)

SEQ_LEN = 512
BATCH_SIZE = 2
MAX_STEPS = 50

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('gpt2')
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '<|pad|>'})

# SFTDataset: instruct по умолчанию (instruction, output)
sft_ds = SFTDataset(
    str(SFT_DATA),
    tokenizer,
    seq_len=SEQ_LEN,
    sft_columns={'format': 'instruct', 'instruction': 'instruction', 'output': 'output'},
)
loader = DataLoader(sft_ds, batch_size=BATCH_SIZE, num_workers=0)
print('SFT dataset ready')

In [ ]:
cfg = HomeConfig(
    vocab_size=len(tokenizer),
    hidden_size=256,
    num_hidden_layers=4,
    num_attention_heads=4,
    max_position_embeddings=SEQ_LEN,
)
model = HomeForCausalLM(cfg)
model.resize_token_embeddings(len(tokenizer))
model = model.to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)
print('params:', sum(p.numel() for p in model.parameters()))

In [ ]:
model.train()
pbar = tqdm(total=MAX_STEPS, desc='SFT')
step = 0
for batch in loader:
    out = model(
        input_ids=batch['input_ids'].to(DEVICE),
        attention_mask=batch.get('attention_mask'),
        labels=batch['labels'].to(DEVICE),
    )
    out.loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    opt.zero_grad(set_to_none=True)
    pbar.set_postfix({'loss': f'{out.loss.item():.4f}'})
    pbar.update(1)
    step += 1
    if step >= MAX_STEPS:
        break
pbar.close()
print('Done. Save: model.save_pretrained("/app/out/sft_playground")')